# Results processing

Load, summarise, compare, and visualise benchmark results.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.patches import Patch
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "06_NOTEBOOKS" else Path.cwd()
MODULES_DIR = REPO_ROOT / "modules"
if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))

from load_results import load_results, save_cache
from xgb_prediction_plots import plot_parity_grid, plot_residual_grid

In [ ]:
DATE = "20260625"
OUTPUTS_ROOT = REPO_ROOT / "07_OUTPUTS"
RUN_DIR = OUTPUTS_ROOT / DATE
FIGURES_DIR = RUN_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_LABELS = {
    "morgan_fps": "ECFP4",
    "hq_descriptor": "QU",
    "chemeleon": "CheMeleon",
    "molclr": "MolCLR",
    "maceoff_large": "MACE-OFF",
}
MODEL_LABELS = {"linear": "LR", "rf": "RF", "xgboost": "XGB", "mlp": "MLP"}
DATASET_LABELS = {"CD": r"$\rho_{cryst}$"}

EMBEDDING_ORDER = ["ECFP4", "QU", "CheMeleon", "MolCLR", "MACE-OFF"]
MODEL_ORDER = ["LR", "RF", "XGB", "MLP"]
DATASET_ORDER = [r"$\rho_{cryst}$", "HOF", "IS"]
DATASET_UNITS = {
    r"$\rho_{cryst}$": r"Test RMSE (g/cm$^3$)",
    "HOF": "Test RMSE (kJ/mol)",
    "IS": "Test RMSE (log(J))",
}
ALPHA = 0.05

## Load and prepare results

In [ ]:
results_df = load_results(OUTPUTS_ROOT, date=DATE)
save_cache(results_df, outputs_root=OUTPUTS_ROOT, date=DATE)
results_df.head()

In [ ]:
def prepare_results(results: pd.DataFrame) -> pd.DataFrame:
    """Apply display labels, ordering, and the agreed quality filter."""
    prepared = results.loc[results["test_R2"] > -0.5].copy()
    prepared["embedding"] = prepared["embedding"].replace(EMBEDDING_LABELS)
    prepared["model"] = prepared["model"].replace(MODEL_LABELS)
    prepared["dataset"] = prepared["dataset"].replace(DATASET_LABELS)

    prepared["embedding"] = pd.Categorical(
        prepared["embedding"], categories=EMBEDDING_ORDER, ordered=True
    )
    prepared["model"] = pd.Categorical(
        prepared["model"], categories=MODEL_ORDER, ordered=True
    )
    prepared["dataset"] = pd.Categorical(
        prepared["dataset"], categories=DATASET_ORDER, ordered=True
    )
    return prepared


df = prepare_results(results_df)
result_columns = [
    "dataset", "model", "seed", "split_method", "embedding",
    "test_MAE", "test_RMSE", "test_R2",
]
clean_results = df[result_columns].dropna(subset=["dataset", "model", "embedding"])
clean_results.head()

## Reusable analysis and plotting helpers

In [ ]:
def save_figure(fig: plt.Figure, filename: str) -> Path:
    """Save a figure consistently, then display it in the notebook."""
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved {path.relative_to(REPO_ROOT)}")
    return path


def select_split(results: pd.DataFrame, split_name: str) -> pd.DataFrame:
    mask = results["split_method"].str.contains(split_name, case=False, na=False)
    return results.loc[mask].copy()


def summarise_results(results: pd.DataFrame) -> pd.DataFrame:
    return (
        results.groupby(["dataset", "embedding"], observed=True)
        [["test_RMSE", "test_R2"]]
        .agg(["mean", "std"])
        .round(3)
    )

In [ ]:
def plot_benchmark_rmse(results: pd.DataFrame, filename: str) -> plt.Figure:
    """Plot test RMSE by prediction head and embedding for each dataset."""
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
    palette = sns.color_palette("tab10", n_colors=len(EMBEDDING_ORDER))
    fig, axes = plt.subplots(len(DATASET_ORDER), 1, figsize=(14, 14), dpi=300)

    pretrained = {"CheMeleon", "MolCLR", "MACE-OFF"}
    first_pretrained = min(EMBEDDING_ORDER.index(name) for name in pretrained)
    last_pretrained = max(EMBEDDING_ORDER.index(name) for name in pretrained)
    group_width = 0.8
    bar_width = group_width / len(EMBEDDING_ORDER)
    legend_handles = legend_labels = None

    for ax, dataset in zip(axes, DATASET_ORDER):
        subset = results.loc[results["dataset"] == dataset]
        sns.barplot(
            data=subset,
            x="model",
            y="test_RMSE",
            hue="embedding",
            hue_order=EMBEDDING_ORDER,
            palette=palette,
            errorbar="sd",
            gap=0.1,
            ax=ax,
        )

        for x_position in ax.get_xticks():
            left = x_position - group_width / 2 + first_pretrained * bar_width
            right = x_position - group_width / 2 + (last_pretrained + 1) * bar_width
            ax.axvspan(left, right, color="lightgray", alpha=0.2, zorder=0)

        for patch in ax.patches:
            patch.set_zorder(2)

        ax.set(title=str(dataset), xlabel="", ylabel=DATASET_UNITS[dataset])
        ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.7)
        sns.despine(ax=ax)

        if legend_handles is None:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    legend_handles.append(Patch(facecolor="lightgray", alpha=0.5))
    legend_labels.append("Pre-trained embeddings")
    fig.legend(
        legend_handles,
        legend_labels,
        title="Embedding",
        loc="center left",
        bbox_to_anchor=(0.84, 0.5),
        frameon=False,
    )
    fig.tight_layout(rect=[0, 0, 0.88, 1])
    save_figure(fig, filename)
    return fig

In [ ]:
def aggregate_paired_runs(results: pd.DataFrame) -> pd.DataFrame:
    """Average over prediction heads while retaining paired experimental runs."""
    return (
        results.loc[results["embedding"].isin(EMBEDDING_ORDER)]
        .groupby(
            ["dataset", "embedding", "seed", "split_method"],
            as_index=False,
            observed=True,
        )["test_RMSE"]
        .mean()
    )


def compare_to_reference(
    results: pd.DataFrame,
    reference: str = "QU",
    candidates: tuple[str, ...] = ("ECFP4", "CheMeleon", "MolCLR", "MACE-OFF"),
) -> pd.DataFrame:
    """Test whether a reference embedding has lower RMSE than each candidate."""
    paired_runs = aggregate_paired_runs(results)
    records = []

    for dataset, dataset_scores in paired_runs.groupby("dataset", observed=True):
        reference_scores = (
            dataset_scores.loc[dataset_scores["embedding"] == reference,
                               ["split_method", "seed", "test_RMSE"]]
            .rename(columns={"test_RMSE": "reference_RMSE"})
        )
        for candidate in candidates:
            candidate_scores = (
                dataset_scores.loc[dataset_scores["embedding"] == candidate,
                                   ["split_method", "seed", "test_RMSE"]]
                .rename(columns={"test_RMSE": "candidate_RMSE"})
            )
            paired = reference_scores.merge(
                candidate_scores, on=["split_method", "seed"], how="inner"
            ).dropna()
            if len(paired) < 2:
                raise ValueError(f"Need at least two paired runs for {dataset=}, {candidate=}")

            test = ttest_rel(
                paired["reference_RMSE"], paired["candidate_RMSE"], alternative="less"
            )
            records.append({
                "Dataset": dataset,
                "Reference": reference,
                "Candidate": candidate,
                "Paired runs": len(paired),
                "Reference mean RMSE": paired["reference_RMSE"].mean(),
                "Candidate mean RMSE": paired["candidate_RMSE"].mean(),
                "t statistic": test.statistic,
                "p value": test.pvalue,
                "Result": (
                    "Reference significantly better"
                    if test.pvalue < ALPHA
                    else "Candidate equal or better"
                ),
            })

    return pd.DataFrame(records).set_index(["Dataset", "Reference", "Candidate"])

In [ ]:
def compare_to_best(results: pd.DataFrame) -> pd.DataFrame:
    """Compare every embedding with the lowest-mean embedding per dataset."""
    paired_runs = aggregate_paired_runs(results)
    records = []

    for dataset, dataset_scores in paired_runs.groupby("dataset", observed=True):
        means = dataset_scores.groupby("embedding", observed=True)["test_RMSE"].mean().sort_values()
        best_embedding = means.index[0]
        best_scores = (
            dataset_scores.loc[dataset_scores["embedding"] == best_embedding,
                               ["split_method", "seed", "test_RMSE"]]
            .rename(columns={"test_RMSE": "best_RMSE"})
        )
        dataset_records = []

        for embedding in means.index[1:]:
            other_scores = (
                dataset_scores.loc[dataset_scores["embedding"] == embedding,
                                   ["split_method", "seed", "test_RMSE"]]
                .rename(columns={"test_RMSE": "other_RMSE"})
            )
            paired = best_scores.merge(
                other_scores, on=["split_method", "seed"], how="inner"
            ).dropna()
            if len(paired) < 2:
                raise ValueError(f"Need at least two paired runs for {dataset=}, {embedding=}")

            test = ttest_rel(paired["best_RMSE"], paired["other_RMSE"], alternative="less")
            dataset_records.append({
                "Dataset": dataset,
                "Best embedding": best_embedding,
                "Compared embedding": embedding,
                "Paired runs": len(paired),
                "Best mean RMSE": paired["best_RMSE"].mean(),
                "Other mean RMSE": paired["other_RMSE"].mean(),
                "RMSE difference": (paired["other_RMSE"] - paired["best_RMSE"]).mean(),
                "t statistic": test.statistic,
                "Raw p value": test.pvalue,
            })

        if dataset_records:
            reject, adjusted_p, _, _ = multipletests(
                [record["Raw p value"] for record in dataset_records],
                alpha=ALPHA,
                method="holm",
            )
            for record, is_worse, p_value in zip(dataset_records, reject, adjusted_p):
                record["Holm-adjusted p value"] = p_value
                record["Result"] = (
                    "Significantly worse than best"
                    if is_worse
                    else "Not significantly worse than best"
                )
            records.extend(dataset_records)

    return pd.DataFrame(records).set_index(
        ["Dataset", "Best embedding", "Compared embedding"]
    )

In [ ]:
def rank_embeddings(results: pd.DataFrame) -> pd.DataFrame:
    """Count dataset/model wins and calculate mean embedding rank."""
    seed_means = (
        results.groupby(["dataset", "model", "embedding"], as_index=False, observed=True)
        ["test_RMSE"]
        .mean()
    )
    seed_means["rank"] = seed_means.groupby(["dataset", "model"], observed=True)[
        "test_RMSE"
    ].rank(method="average", ascending=True)
    seed_means["win"] = seed_means["rank"].eq(1)

    return (
        seed_means.groupby("embedding", observed=True)
        .agg(**{"Win Count": ("win", "sum"), "Avg Rank": ("rank", "mean")})
        .sort_values(["Win Count", "Avg Rank"], ascending=[False, True])
        .round(2)
    )

## Random splits

In [ ]:
random_results = select_split(clean_results, "random")
display(summarise_results(random_results))
plot_benchmark_rmse(random_results, "random_split_benchmark_rmse.png");

### Are the foundation-model embeddings equal to or better than QU?

One-sided paired t-tests use matched seed/split runs after averaging across prediction heads. Lower RMSE is better.

In [ ]:
display(compare_to_reference(random_results).round(3))

### Comparisons with the best embedding

Each embedding is compared with the lowest-mean embedding for its dataset. Holm correction is applied within each dataset.

In [ ]:
display(compare_to_best(random_results).round(3))
display(rank_embeddings(random_results))

## Cluster splits

In [ ]:
cluster_results = select_split(clean_results, "cluster")
display(summarise_results(cluster_results))
plot_benchmark_rmse(cluster_results, "cluster_split_benchmark_rmse.png");

In [ ]:
display(compare_to_best(cluster_results).round(3))
display(rank_embeddings(cluster_results))

## Penalty from harder splits

In [ ]:
def calculate_split_penalties(results: pd.DataFrame) -> pd.DataFrame:
    group_columns = ["dataset", "embedding", "model"]
    random_means = (
        select_split(results, "random_seed")
        .groupby(group_columns, as_index=False, observed=True)["test_RMSE"]
        .mean()
        .rename(columns={"test_RMSE": "random_RMSE"})
    )
    cluster_means = (
        select_split(results, "cluster")
        .groupby(group_columns, as_index=False, observed=True)["test_RMSE"]
        .mean()
        .rename(columns={"test_RMSE": "cluster_RMSE"})
    )
    penalties = random_means.merge(cluster_means, on=group_columns, how="inner")
    penalties["normalized_split_penalty"] = (
        penalties["cluster_RMSE"] - penalties["random_RMSE"]
    ) / penalties["random_RMSE"]
    return penalties


def plot_split_penalties(penalties: pd.DataFrame) -> plt.Figure:
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.5)
    fig, axes = plt.subplots(1, len(DATASET_ORDER), figsize=(14, 4), sharey=True, dpi=300)

    for ax, dataset in zip(np.atleast_1d(axes), DATASET_ORDER):
        subset = penalties.loc[penalties["dataset"] == dataset]
        sns.pointplot(
            data=subset,
            x="embedding",
            y="normalized_split_penalty",
            hue="embedding",
            order=EMBEDDING_ORDER,
            palette=sns.color_palette("tab10", n_colors=len(EMBEDDING_ORDER)),
            errorbar="sd",
            markers="o",
            linestyles="none",
            legend=False,
            ax=ax,
        )
        ax.set(
            title=str(dataset),
            xlabel="",
            ylabel="Normalized split penalty" if ax is axes[0] else "",
        )
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.7)
        sns.despine(ax=ax)

    fig.tight_layout()
    save_figure(fig, "normalized_cluster_split_penalty.png")
    return fig


split_penalties = calculate_split_penalties(df)
plot_split_penalties(split_penalties);

In [ ]:
display(
    split_penalties.groupby(["dataset", "embedding"], observed=True)
    [["normalized_split_penalty"]]
    .agg(["mean", "std"])
    .round(3)
)

## Impact of the prediction head

In [ ]:
def plot_prediction_head_ranks(results: pd.DataFrame) -> plt.Figure:
    summary = (
        results.groupby(
            ["dataset", "embedding", "model", "split_method"],
            as_index=False,
            observed=True,
        )["test_RMSE"]
        .agg(mean_RMSE="mean")
    )
    summary["rank"] = summary.groupby(
        ["dataset", "embedding", "split_method"], observed=True
    )["mean_RMSE"].rank(method="dense", ascending=True)

    heatmap_data = (
        summary.groupby(["embedding", "model"], observed=True)["rank"]
        .mean()
        .unstack("model")
        .reindex(index=EMBEDDING_ORDER, columns=MODEL_ORDER)
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".2f",
        cmap="viridis_r",
        linewidths=0.5,
        cbar_kws={"label": "Mean RMSE rank"},
        ax=ax,
    )
    ax.set(xlabel="Prediction head", ylabel="Embedding")
    ax.tick_params(axis="both", rotation=0)
    fig.tight_layout()
    save_figure(fig, "prediction_head_mean_rmse_rank.png")
    return fig


plot_prediction_head_ranks(random_results);

## Prediction parity and residuals

In [ ]:
prediction_script = REPO_ROOT / "05_PY_SCRIPTS" / "generate_xgb_predictions.py"
subprocess.run([sys.executable, str(prediction_script)], check=True)

predictions_path = REPO_ROOT / "deep_dive_xgb_predictions.parquet"
predictions_df = pd.read_parquet(predictions_path)
predictions_df["embedding"] = predictions_df["embedding"].replace(EMBEDDING_LABELS)
predictions_df["dataset"] = predictions_df["dataset"].replace(DATASET_LABELS)
predictions_df["dataset"] = pd.Categorical(
    predictions_df["dataset"], categories=DATASET_ORDER, ordered=True
)
predictions_df.head()

In [ ]:
fig = plot_residual_grid(
    predictions_df,
    embeddings=["QU", "MACE-OFF"],
    show_pred_std=True,
    use_aggregated=True,
    figsize=(6, 10),
)
save_figure(fig, "xgb_prediction_residuals.png");

In [ ]:
fig = plot_parity_grid(
    predictions_df,
    embeddings=["QU", "MACE-OFF"],
    use_aggregated=True,
    color_by=None,
    show_pred_std=True,
)
save_figure(fig, "xgb_prediction_parity.png");